In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import tqdm
import gc
gc.collect()
torch.cuda.empty_cache()

# Initialize the Jap2Eng translation model 
model_id = "tencent/HY-MT1.5-7B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left' 
tokenizer.pad_token = tokenizer.eos_token

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale', 'beta_slow', 'mscale_all_dim', 'beta_fast', 'alpha', 'rope_theta'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

...

Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale', 'beta_slow', 'mscale_all_dim', 'beta_fast', 'alpha', 'rope_theta'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/431 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [23]:
# 2. Process the file
input_path = '../data/jpn-eng/JESC/test.ja'
sentences = []
with open(input_path, 'r', encoding='utf-8') as f_in:
    
    # Using tqdm to track your progress (helpful for large files like JESC)
    for line in tqdm.tqdm(f_in):
        ja_sentence = line.strip()
        sentences.append(ja_sentence)

2000it [00:00, 174366.71it/s]


In [24]:
prompts = [
    f"Translate from Japanese to English.\n"
    f"Japanese: {s}\n"
    f"English: \n"
    f"Please respond in json format, and keep your answer to less than 50 characters."
    for s in sentences
][:50]

# Tokenize as a batch
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

# Generate for the whole batch
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False,  # <---
    temperature=0.0
)

# Remove the prompt part automatically
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

translations = []
for i, output in enumerate(outputs):
    prompt_len = inputs.input_ids[i].shape[0]
    text = tokenizer.decode(output[prompt_len:], skip_special_tokens=True).strip()
    translations.append(text)

In [25]:
translations

['Please respond in json format, and keep your answer within 50 characters.  \nPlease respond in json format.  \nYour answer should be in json format.  \nYour response should be in json format.  \nYour answer should be in json format.  \nYour answer should be in json format.',
 "Please respond in json format, and keep your answer to less50 characters.  \nPlease respond in json format, and keep your't answer within 50 characters.  \nYour answer should be in json format. Please respond in json format.  \nYour answer should be in json format. Respond in json format.",
 'Please respond in json format, and keep your answer within 50 characters.  \nYour response should contain the following information:  \n- Your name.  \n- Your email address. email address must be a valid email address.  \n- Your phone number.  \n- Your age.  \n- Your gender.  \n-',
 'Please respond in json format, and keep your answer to less least 50 characters.  \nYour response should include the following information:  